<a href="https://colab.research.google.com/github/obrnocsta/festival-musica/blob/main/1-desafio/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Parte I - Conexão e Testes

**01 - Instalar**

In [ ]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

**02 - Importações e Autenticação (service account)**

In [ ]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@graceful-wall-513817-f9.iam.gserviceaccount.com


**03 - Abrir a Planilha**

In [ ]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "1DshIt2ENx4LqhdScI_9JC0KtDfukfFaFAutHND8QRbM"
NOME_TABELA = "atracoes"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1

# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Festival_Musica | atracoes | https://docs.google.com/spreadsheets/d/1DshIt2ENx4LqhdScI_9JC0KtDfukfFaFAutHND8QRbM


**04 - Cabeçalho - Tabela Dimensão (clientes)**

In [ ]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["atracao_id", "nome_artista_ficticio", "genero_musical", "edicao_festival", "dia_apresentacao", "palco", "horario_inicio"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


**05 - 10 linhas de teste - Tabela Dimensão (clientes)**

In [ ]:
from datetime import date, timedelta
import random

from faker import Faker

# Configuração dos dados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(90)
random.seed(90)

# Use 10 para cumprir o Passo 9.
# Depois, você pode alterar para 100, 1000 ou 10100.
QUANTIDADE = 10

COLUNAS = [
    "atracao_id",
    "nome_artista_ficticio",
    "genero_musical",
    "edicao_festival",
    "dia_apresentacao",
    "palco",
    "horario_inicio",
]

GENEROS = [
    "pop",
    "rock",
    "eletronica",
    "sertanejo",
    "funk",
    "mpb",
    "rap",
    "forró",
    "reggae",
]

PESOS_GENEROS = [
    16,
    14,
    14,
    14,
    12,
    10,
    10,
    5,
    5,
]

PALCOS = [
    "principal",
    "secundario",
    "eletronico",
    "acustico",
]

PESOS_PALCOS = [
    30,
    35,
    20,
    15,
]

TEMAS_90 = [
    "VHS",
    "Neon",
    "Arcade",
    "Laser",
    "Mutante",
    "Alien",
    "Zumbi",
    "Turbo",
    "Cassete",
    "Monstro",
    "Pixel",
    "Apocalipse",
]

TERMOS_MUSICAIS = [
    "Beat",
    "Band",
    "Crew",
    "Wave",
    "Groove",
    "Mix",
    "Sound",
    "Power",
    "Stars",
    "Noise",
]

def nome_artista():
    """Cria um nome fictício com estilo musical dos anos 1990."""
    prefixo = random.choice(
        ["Os", "As", "DJ", "Banda", "Coletivo", ""]
    )
    nome = f"{random.choice(TEMAS_90)} {random.choice(TERMOS_MUSICAIS)}"
    return f"{prefixo} {nome}".strip()

def gerar_edicoes():
    """Gera 48 edições, seis por ano, entre 1990 e 1997."""
    edicoes = []

    for ano in range(1990, 1998):
        for mes in [1, 3, 5, 7, 9, 11]:
            inicio = date(ano, mes, random.randint(1, 20))

            nome = random.choice(
                [
                    "VHS",
                    "Neon",
                    "Arcade",
                    "Trash",
                    "Laser",
                    "Mutante",
                    "Turbo",
                    "Apocalipse",
                ]
            )

            edicoes.append(
                {
                    "nome": f"Festival {nome} {ano}",
                    "inicio": inicio,
                }
            )

    return edicoes

def gerar_horarios():
    """Cria horários de 12:00 a 23:45, de 15 em 15 minutos."""
    return [
        f"{hora:02d}:{minuto:02d}"
        for hora in range(12, 24)
        for minuto in range(0, 60, 15)
    ]

def gerar_atracoes(quantidade):
    """Gera a quantidade solicitada de atrações fictícias."""
    edicoes = gerar_edicoes()
    horarios = gerar_horarios()
    vagas = []

    for edicao in edicoes:
        for dia in range(3):
            data_show = (
                edicao["inicio"] + timedelta(days=dia)
            ).isoformat()

            for palco in PALCOS:
                for horario in random.sample(
                    horarios,
                    len(horarios),
                ):
                    vagas.append(
                        {
                            "edicao_festival": edicao["nome"],
                            "dia_apresentacao": data_show,
                            "palco": palco,
                            "horario_inicio": horario,
                        }
                    )

    if quantidade > len(vagas):
        raise ValueError(
            f"Quantidade máxima disponível: {len(vagas)}"
        )

    random.shuffle(vagas)

    linhas = []

    for identificador, vaga in enumerate(
        vagas[:quantidade],
        start=1,
    ):
        linhas.append(
            {
                "atracao_id": identificador,
                "nome_artista_ficticio": nome_artista(),
                "genero_musical": random.choices(
                    GENEROS,
                    weights=PESOS_GENEROS,
                    k=1,
                )[0],
                "edicao_festival": vaga["edicao_festival"],
                "dia_apresentacao": vaga["dia_apresentacao"],
                "palco": vaga["palco"],
                "horario_inicio": vaga["horario_inicio"],
            }
        )

    return linhas

# Apaga os dados anteriores e preserva uma aba limpa.
ws.clear()

# Escreve o cabeçalho.
ws.update(
    range_name="A1",
    values=[COLUNAS],
)

# Gera registros.
linhas = gerar_atracoes(QUANTIDADE)

# Organiza os valores conforme a ordem do cabeçalho.
valores = [
    [linha[coluna] for coluna in COLUNAS]
    for linha in linhas
]

# Insere os dados abaixo do cabeçalho.
ws.append_rows(
    valores,
    value_input_option="RAW",
)

print(len(valores), "linhas enviadas")

10 linhas enviadas


**06 - Conferência – Tabela Dimensão (clientes)**

In [ ]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()

# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 11
Cabeçalho confere: True
[['atracao_id', 'nome_artista_ficticio', 'genero_musical', 'edicao_festival', 'dia_apresentacao', 'palco', 'horario_inicio'], ['1', 'DJ Zumbi Mix', 'eletronica', 'Festival Turbo 1995', '1995-05-20', 'acustico', '21:15'], ['2', 'Banda Apocalipse Noise', 'mpb', 'Festival Arcade 1991', '1991-09-19', 'acustico', '23:30']]


## Parte II - Base Completa

**07 - Funções de envio de dados - Planilha**

In [ ]:
# Importa recursos para controlar pausas, acessar a planilha e tratar tabelas.
import time
import gspread
import pandas as pd

def df_para_valores(df):
    """DataFrame -> lista de listas com tipos nativos do Python."""

    # Trabalha em uma cópia para não alterar o DataFrame original.
    df = df.copy()
    for c in df.columns:
        # Converte datas para texto em um formato padronizado.
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.strftime("%Y-%m-%d %H:%M:%S")
        # Converte outros valores de data e hora para texto.
        elif df[c].dtype == object:
            df[c] = df[c].map(
                lambda v: v.isoformat() if hasattr(v, "isoformat") else v)
    # Substitui valores ausentes por texto vazio e retorna linhas de valores.
    return df.astype(object).where(df.notna(), "").values.tolist()

def enviar(sh, df, nome_aba, lote=5000, pausa=2):
    """Escreve cabeçalho e linhas numa aba, em lotes, e confere a contagem."""

    # Formata números com ponto como separador de milhar.
    n = lambda x: f"{x:,}".replace(",", ".")
    try:
        # Reutiliza a aba existente e apaga seu conteúdo anterior.
        ws = sh.worksheet(nome_aba)
        ws.clear()
    except gspread.WorksheetNotFound:
        # Cria uma aba caso ela ainda não exista.
        ws = sh.add_worksheet(title=nome_aba, rows=len(df) + 1,
                              cols=len(df.columns))

    # Ajusta o tamanho da aba e grava os nomes das colunas.
    ws.resize(rows=len(df) + 1, cols=len(df.columns))
    ws.update(range_name="A1", values=[list(df.columns)])

    # Converte o DataFrame e envia as linhas em lotes.
    valores = df_para_valores(df)
    for i in range(0, len(valores), lote):
        ws.append_rows(valores[i:i + lote], value_input_option="RAW")
        print(f"  {nome_aba}: {n(min(i + lote, len(valores)))} / {n(len(valores))}")
        time.sleep(pausa)               # respeita o limite de escritas/minuto

    # Confere se a quantidade de linhas gravadas corresponde à esperada.
    lidas = len(ws.col_values(1)) - 1   # lê de volta e confere

    if lidas != len(df):
        raise RuntimeError(f"Aba {nome_aba}: enviei {len(df)}, li {lidas}.")
    print(f"OK {nome_aba}: {n(len(df))} linhas na planilha")

def remover_aba_padrao(sh):
    """Apaga a aba vazia que o Google cria junto com a planilha nova."""

    # Localiza e remove a aba padrão somente se houver outra aba disponível.
    abas = sh.worksheets()
    padrao = ("Página1", "Sheet1", "Planilha1")

    for w in abas:
        if w.title in padrao and len(abas) > 1 and not w.get_all_values():
            sh.del_worksheet(w)

**08 - Carregar Bases de dados - Prontas**

In [ ]:
# Abre a opção de selecionar arquivos CSV no Colab e envia os arquivos.
from google.colab import files
files.upload()   # selecione os arquivos CSV do seu tema

# Carrega cada CSV em um DataFrame e associa-o ao nome da tabela.
# Altere o nome da tabela de acordo com a sua Base de dados
BASES = {
    "atracoes": pd.read_csv("atracoes.csv"),
    "ingressos_vendidos": pd.read_csv("ingressos_vendidos.csv"),
}

# Calcula e exibe a quantidade de linhas carregadas.
total = sum(len(d) for d in BASES.values())
print({n: len(d) for n, d in BASES.items()}, "| total:", total)

Saving atracoes.csv to atracoes (1).csv
Saving ingressos_vendidos.csv to ingressos_vendidos (1).csv
{'atracoes': 10100, 'ingressos_vendidos': 55000} | total: 65100


**09 - Enviar para a planilha em lotes**

In [ ]:
# Envia cada tabela para uma aba e remove a aba padrão, se estiver vazia.
for nome, df in BASES.items():
    enviar(sh, df, nome)        # uma aba por tabela
remover_aba_padrao(sh)

  atracoes: 5.000 / 10.100
  atracoes: 10.000 / 10.100
  atracoes: 10.100 / 10.100
OK atracoes: 10.100 linhas na planilha
  ingressos_vendidos: 5.000 / 55.000
  ingressos_vendidos: 10.000 / 55.000
  ingressos_vendidos: 15.000 / 55.000
  ingressos_vendidos: 20.000 / 55.000
  ingressos_vendidos: 25.000 / 55.000
  ingressos_vendidos: 30.000 / 55.000
  ingressos_vendidos: 35.000 / 55.000
  ingressos_vendidos: 40.000 / 55.000
  ingressos_vendidos: 45.000 / 55.000
  ingressos_vendidos: 50.000 / 55.000
  ingressos_vendidos: 55.000 / 55.000
OK ingressos_vendidos: 55.000 linhas na planilha


**10 - Validar a base na planilha**

In [ ]:
# Soma e exibe a quantidade total de linhas das tabelas.
total = sum(len(d) for d in BASES.values())
print("Total de linhas:", f"{total:,}".replace(",", "."))

# Interrompe a execução se o volume mínimo não for atingido.
assert total >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

# Confere os cabeçalhos e a quantidade de registros em cada aba.
for nome, df in BASES.items():
    w = sh.worksheet(nome)
    assert w.row_values(1) == list(df.columns), f"Cabeçalho diferente em {nome}"
    assert len(w.col_values(1)) - 1 == len(df), f"Contagem diferente em {nome}"

# Confirma que todas as verificações passaram.
print("A planilha confere com a base gerada.")

Total de linhas: 65.100
A planilha confere com a base gerada.
